In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------------------------------------------
# 1. Load and Clean Dataset
# -------------------------------------------------------------
filepath = r'C:\Users\dell\OneDrive\Desktop\KMEC\PS Project\432_2024_5740_MOESM3_ESM.xlsx'

# Read sheet skipping non-data headers
raw_df = pd.read_excel(filepath, sheet_name='Results')
raw_df.columns = raw_df.iloc[0]
df = raw_df.iloc[1:].copy().reset_index(drop=True)

# Keep only rows with a valid target pair separated by '+'
df = df[df['Target(Gene Name)'].str.contains(r'\+', na=False)].copy()

# =============================================================
# 2. CLEAN COLUMN NAMES
# =============================================================

df.columns = df.columns.astype(str).str.strip()

print("\nAvailable columns:")
print(df.columns.tolist())


# =============================================================
# 3. CHECK REQUIRED COLUMNS
# =============================================================

required_columns = [
    'Target(Gene Name)',
    'Drug Highest Phase'
]

for column in required_columns:
    if column not in df.columns:
        raise ValueError(
            f"Required column '{column}' was not found in the dataset."
        )


# =============================================================
# 4. CLEAN TARGET PAIR COLUMN
# =============================================================

df['Target(Gene Name)'] = (
    df['Target(Gene Name)']
    .astype(str)
    .str.strip()
)

invalid_targets = [
    '',
    '-',
    'nan',
    'None',
    'NA',
    'N/A'
]

df = df[
    ~df['Target(Gene Name)'].isin(invalid_targets)
].copy()


# =============================================================
# 5. SPLIT TARGET PAIRS
# =============================================================

def split_and_sort_targets(target_string):

    parts = target_string.split('+')

    parts = [
        part.strip().upper()
        for part in parts
        if part.strip()
    ]

    if len(parts) != 2:
        return pd.Series([np.nan, np.nan])

    # Make pair order-independent
    parts.sort()

    return pd.Series([
        parts[0],
        parts[1]
    ])


df[['Target_A', 'Target_B']] = (
    df['Target(Gene Name)']
    .apply(split_and_sort_targets)
)


# Remove invalid pairs
df = df.dropna(
    subset=['Target_A', 'Target_B']
).reset_index(drop=True)


# =============================================================
# 6. CREATE APPROVAL LABEL
# =============================================================

# 1 = Approved
# 0 = Not Approved

def create_label(value):

    if pd.isna(value):
        return 0

    value = str(value).strip().lower()

    if 'approved' in value:
        return 1

    return 0


df['Target_Label'] = (
    df['Drug Highest Phase']
    .apply(create_label)
)


# =============================================================
# 7. DISPLAY DATA DISTRIBUTION
# =============================================================

print("\n==========================================")
print("DATASET INFORMATION")
print("==========================================")

print(
    f"Total valid target-pair records: {len(df)}"
)

print("\nTarget label distribution:")
print(
    df['Target_Label'].value_counts()
)

print("\nTarget label percentage:")
print(
    df['Target_Label']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


# =============================================================
# 8. REMOVE DUPLICATE TARGET PAIRS
# =============================================================

# If the same pair appears multiple times,
# the pair is considered approved if at least
# one record is approved.

pair_labels = (
    df.groupby(
        ['Target_A', 'Target_B']
    )['Target_Label']
    .max()
    .reset_index()
)


print("\n==========================================")
print("UNIQUE TARGET PAIRS")
print("==========================================")

print(
    f"Unique target pairs: {len(pair_labels)}"
)

print("\nUnique pair label distribution:")
print(
    pair_labels['Target_Label']
    .value_counts()
)


# =============================================================
# 9. CHECK CLASS AVAILABILITY
# =============================================================

if pair_labels['Target_Label'].nunique() < 2:
    raise ValueError(
        "The dataset contains only one class after preprocessing. "
        "XGBoost classification requires both 0 and 1 classes."
    )


# =============================================================
# 10. CREATE X AND y
# =============================================================

X = pair_labels[
    ['Target_A', 'Target_B']
]

y = pair_labels[
    'Target_Label'
]


# =============================================================
# 11. ONE-HOT ENCODING
# =============================================================

preprocessor = ColumnTransformer(
    transformers=[

        (
            'target_features',

            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False
            ),

            ['Target_A', 'Target_B']
        )
    ]
)


# =============================================================
# 12. CLASS IMBALANCE
# =============================================================

negative_count = (
    y == 0
).sum()

positive_count = (
    y == 1
).sum()


if positive_count == 0:
    raise ValueError(
        "No positive/approved examples were found."
    )


scale_pos_weight = (
    negative_count / positive_count
)


print("\n==========================================")
print("CLASS BALANCING")
print("==========================================")

print(
    f"Negative samples: {negative_count}"
)

print(
    f"Positive samples: {positive_count}"
)

print(
    f"scale_pos_weight: {scale_pos_weight:.4f}"
)


# =============================================================
# 13. XGBOOST CLASSIFIER
# =============================================================

model = xgb.XGBClassifier(

    n_estimators=150,

    learning_rate=0.05,

    max_depth=3,

    min_child_weight=2,

    subsample=0.8,

    colsample_bytree=0.8,

    scale_pos_weight=scale_pos_weight,

    eval_metric='logloss',

    random_state=42,

    tree_method='hist'
)


# =============================================================
# 14. COMPLETE PIPELINE
# =============================================================

pipeline = Pipeline(
    steps=[

        (
            'preprocessor',
            preprocessor
        ),

        (
            'classifier',
            model
        )
    ]
)


# =============================================================
# 15. STRATIFIED CROSS-VALIDATION
# =============================================================

class_counts = y.value_counts()

minimum_class_count = class_counts.min()

if minimum_class_count < 5:

    print(
        "\nWARNING:"
        "\nThere are fewer than 5 samples in the smallest class."
        "\n5-fold cross-validation may not be reliable."
    )

    n_splits = max(
        2,
        minimum_class_count
    )

else:

    n_splits = 5


skf = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=42
)


cv_roc_auc = cross_val_score(
    pipeline,
    X,
    y,
    cv=skf,
    scoring='roc_auc'
)


print("\n==========================================")
print("CROSS-VALIDATION RESULTS")
print("==========================================")

print(
    f"{n_splits}-Fold ROC-AUC:"
)

print(
    f"{cv_roc_auc.mean():.4f}"
)

print(
    f"Standard deviation:"
    f" {cv_roc_auc.std():.4f}"
)


# =============================================================
# 16. FIT FINAL MODEL
# =============================================================

pipeline.fit(
    X,
    y
)


# =============================================================
# 17. PREDICTION FUNCTION
# =============================================================

def predict_pair(
    target_1: str,
    target_2: str
):

    target_1 = target_1.strip().upper()
    target_2 = target_2.strip().upper()

    if not target_1 or not target_2:

        raise ValueError(
            "Both target names must be provided."
        )

    # Make pair order-independent
    pair = sorted([
        target_1,
        target_2
    ])

    input_df = pd.DataFrame({

        'Target_A': [pair[0]],

        'Target_B': [pair[1]]
    })

    prediction = pipeline.predict(
        input_df
    )[0]

    probability = pipeline.predict_proba(
        input_df
    )[0][1]

    return {

        'Target_A': pair[0],

        'Target_B': pair[1],

        'Prediction': int(prediction),

        'Approval_Probability':
            round(
                float(probability),
                4
            )
    }


# =============================================================
# 18. USER INPUT
# =============================================================

if __name__ == "__main__":

    print("\n==========================================")
    print("BISPECIFIC TARGET-PAIR APPROVAL PREDICTION")
    print("==========================================")

    user_target_1 = input(
        "\nEnter Target 1 (e.g., BCMA): "
    )

    user_target_2 = input(
        "Enter Target 2 (e.g., CD3): "
    )

    result = predict_pair(
        user_target_1,
        user_target_2
    )

    print("\n==========================================")
    print("PREDICTION RESULT")
    print("==========================================")

    print(
        f"Target Pair: "
        f"{result['Target_A']} + "
        f"{result['Target_B']}"
    )

    print(
        f"Output: {result['Prediction']}"
    )

    if result['Prediction'] == 1:

        print(
            "Meaning: Approved"
        )

    else:

        print(
            "Meaning: Not Approved"
        )

    print(
        f"Approval Probability: "
        f"{result['Approval_Probability']}"
    )


Available columns:
['Drug', 'Target(Gene Name)', 'Target(Gene Symbol)', 'Active Indication(Indication Name)', 'Originator Organization', 'Drug Highest Phase', 'Drug Approved Country/Location', 'Approval Date']

DATASET INFORMATION
Total valid target-pair records: 782

Target label distribution:
Target_Label
0    770
1     12
Name: count, dtype: int64

Target label percentage:
Target_Label
0    98.47
1     1.53
Name: proportion, dtype: float64

UNIQUE TARGET PAIRS
Unique target pairs: 337

Unique pair label distribution:
Target_Label
0    329
1      8
Name: count, dtype: int64

CLASS BALANCING
Negative samples: 329
Positive samples: 8
scale_pos_weight: 41.1250

CROSS-VALIDATION RESULTS
5-Fold ROC-AUC:
0.4425
Standard deviation: 0.2074

BISPECIFIC TARGET-PAIR APPROVAL PREDICTION

PREDICTION RESULT
Target Pair: BCMA + MAS
Output: 0
Meaning: Not Approved
Approval Probability: 0.0657
